# Feature engineering — Step 9

**CSE437 Data Science | Group 15 | Owner: Sadat | Next: Step 10**

Step 9 implements eight deterministic derived fields with an explicit extended preprocessing schema. Statistical feature selection, dimensionality reduction, and predictive comparisons remain pending for later stages.

The original proposal, dataset, target, questions, cohort and CV assignments remain unchanged. No target values are read by this notebook, and no held-out rows are fitted or transformed. Read `report/step9_feature_engineering.md` for decisions and limitations.

**Execution provenance:** all five code cells below were executed sequentially in a fresh Python process with actual stdout captured. Jupyter, IPython and nbformat were unavailable in this runtime. JSON structure checks do not substitute for canonical nbformat validation or the required fresh Jupyter-kernel run; those remain final submission gates. No outputs are illustrative or fabricated.


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data/splits/step6_split_plan.json").is_file(), "Run from the repo root or notebooks directory."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.feature_engineering import BookingFeatureEngineer, make_feature_preprocessor, DERIVED_COLUMNS
from src.feature_audit import run_feature_audit
from src.preprocessing import LOG_COLUMNS, NUMERIC_COLUMNS, CATEGORICAL_COLUMNS
print("Step 9: fixed features; training-fold imputation and encoding; test untouched.")


Step 9: fixed features; training-fold imputation and encoding; test untouched.


## Feature decisions

| Derived field | Formula / interpretation |
| --- | --- |
| `total_nights` | Weekend nights + weekday nights; zero stays retained |
| `total_guests` | Adults + children + babies; unknown components propagate |
| `previous_bookings_total` | Previous cancellations + previous noncanceled bookings |
| `has_booking_history` | 1 if history total > 0, 0 if zero, missing if unknown |
| `previous_cancellation_share` | Cancellations / history total; zero for no history, interpreted together with the history flag |
| `company_code_recorded` | 1 if company code is present, 0 if null; literal code 0 is present |
| `arrival_month_sin` | sin(2π(month−1)/12) |
| `arrival_month_cos` | cos(2π(month−1)/12) |

Month names are replaced with the two fixed calendar coordinates; other source fields are retained for later selection. Company IDs remain excluded. The company flag represents code recording, not verified payment or booking-time availability. Unknown or invalid calendar month names fail explicitly.

Totals and the ratio are formed **before** training-fold imputation. The pipeline logs the three nonnegative totals along with Step 7's four log fields. Ratios, flags and cyclic coordinates are not logged. Separate median-imputed totals need not equal sums of imputed components. Fixed missing indicators retain children, ADR, total nights, total guests and cancellation-share missingness.


In [2]:
summary, statistics = run_feature_audit(ROOT)
print(json.dumps({k: summary[k] for k in ["development_rows", "retained_source_fields", "derived_fields",
    "fields_before_encoding", "fixed_missing_indicators", "development_zero_night_bookings_retained",
    "development_unknown_total_guests", "development_no_recorded_history", "development_company_code_recorded",
    "test_rows_fitted_or_transformed", "target_values_read", "predictive_models_trained"]}, indent=2))


{
  "development_rows": 95415,
  "retained_source_fields": 24,
  "derived_fields": [
    "total_nights",
    "total_guests",
    "previous_bookings_total",
    "has_booking_history",
    "previous_cancellation_share",
    "company_code_recorded",
    "arrival_month_sin",
    "arrival_month_cos"
  ],
  "fields_before_encoding": 32,
  "fixed_missing_indicators": [
    "children",
    "adr",
    "total_nights",
    "total_guests",
    "previous_cancellation_share"
  ],
  "development_zero_night_bookings_retained": 604,
  "development_unknown_total_guests": 4,
  "development_no_recorded_history": 86585,
  "development_company_code_recorded": 5900,
  "test_rows_fitted_or_transformed": 0,
  "target_values_read": false,
  "predictive_models_trained": 0
}


## Observed development feature values

These are fixed-formula diagnostics, not performance results or fitted full-development transformations. Missing values below precede imputation.


In [3]:
print(statistics.round(4).to_string(index=False))


                    feature   count    mean    std  min    25%  50%   75%  max  missing_before_imputation
               total_nights 95415.0  3.3562 2.5564  0.0  2.000  3.0 4.000 69.0                          0
               total_guests 95411.0  1.9434 0.7259  1.0  2.000  2.0 2.000 55.0                          4
    previous_bookings_total 95415.0  0.2395 1.7702  0.0  0.000  0.0 0.000 67.0                          0
        has_booking_history 95415.0  0.0925 0.2898  0.0  0.000  0.0 0.000  1.0                          0
previous_cancellation_share 95415.0  0.0625 0.2404  0.0  0.000  0.0 0.000  1.0                          0
      company_code_recorded 95415.0  0.0618 0.2409  0.0  0.000  0.0 0.000  1.0                          0
          arrival_month_sin 95415.0 -0.0484 0.7384 -1.0 -0.866  0.0 0.866  1.0                          0
          arrival_month_cos 95415.0 -0.0066 0.6726 -1.0 -0.500 -0.0 0.500  1.0                          0


## Training-fold verification

Each variant is fitted separately on the training prefix. Validation transformation must not change fitted state. Vocabulary, medians and scaling are checked against that training prefix; no test data enter the checks.


In [4]:
fold_table = pd.DataFrame([{
    "fold": f["fold"], "train_rows": f["training"]["rows"], "validation_rows": f["validation"]["rows"],
    "encoded_columns": f["training"]["columns"], "validation_nonfinite": f["validation"]["nonfinite_values"],
    "unseen_month_rows": f["validation_rows_with_unseen_month"],
    "validation_kept_fitted_state": f["validation_did_not_change_fitted_state"]
} for f in summary["folds"]])
print(fold_table.to_string(index=False))
print("\nMonth names absent from training:")
for fold in summary["folds"]:
    print(fold["fold"], fold["validation_months_absent_from_training"])
assert summary["fields_before_encoding"] == 32
assert all(f["scaled_and_unscaled_variants_verified"] for f in summary["folds"])
assert all(f["validation"]["nonfinite_values"] == 0 for f in summary["folds"])
print("\nBoth model-compatible variants passed all three frozen folds.")


 fold  train_rows  validation_rows  encoded_columns  validation_nonfinite  unseen_month_rows  validation_kept_fitted_state
    1       23797            23893              332                     0              23475                          True
    2       47690            23776              421                     0                  0                          True
    3       71466            23949              490                     0                  0                          True

Month names absent from training:
1 ['April', 'February', 'June', 'March', 'May']
2 []
3 []

Both model-compatible variants passed all three frozen folds.


## Synthetic examples for explanation

The following three rows are invented demonstrations of formulas, not source records. No-history and observed zero cancellation share have different history flags. A missing child count leaves the guest total missing until fold-fitted imputation.


In [5]:
row = {c: 1 for c in LOG_COLUMNS + NUMERIC_COLUMNS}
row.update({c: "A" for c in CATEGORICAL_COLUMNS})
row.update(arrival_date_month="January", company=np.nan, agent=1, adults=2, babies=0,
           assigned_room_type="A", booking_changes=0, days_in_waiting_list=0)
examples = pd.DataFrame([row.copy() for _ in range(3)])
examples["children"] = [0, np.nan, 1]
examples["previous_cancellations"] = [0, 0, 2]
examples["previous_bookings_not_canceled"] = [0, 4, 6]
examples["company"] = [np.nan, 0, 42]
examples["arrival_date_month"] = ["December", "January", "June"]
examples["stays_in_weekend_nights"] = [0, 1, 2]
examples["stays_in_week_nights"] = [0, 4, 3]
result = BookingFeatureEngineer().fit_transform(examples)
assert result.previous_cancellation_share.tolist() == [0, 0, .25]
assert result.has_booking_history.tolist() == [0, 1, 1]
assert np.isnan(result.total_guests.iloc[1])
print(result[list(DERIVED_COLUMNS)].round(4).to_string(index=False))


 total_nights  total_guests  previous_bookings_total  has_booking_history  previous_cancellation_share  company_code_recorded  arrival_month_sin  arrival_month_cos
          0.0           2.0                      0.0                  0.0                         0.00                    0.0               -0.5              0.866
          5.0           NaN                      4.0                  1.0                         0.00                    1.0                0.0              1.000
          5.0           3.0                      8.0                  1.0                         0.25                    1.0                0.5             -0.866


## Handoff to Step 10 — Sadat

The 32 pre-encoding fields are a **candidate set**, not the final selected features. Demonstrate both statistical feature selection and dimensionality reduction within the frozen development folds. Use a new `make_feature_preprocessor()` in each candidate model pipeline; never reuse a matrix prefitted on all development rows. Fold indices from `development_cv(assignments)` are relative to development rows in Step 5 source order.

The fixed month representation avoids missing one-hot categories but does not supply evidence about unseen seasons. The totals are redundant with components by design; assess redundancy and usefulness later. Company/deposit/history timing remains a retrospective limitation. No predictive performance improvement is claimed.

Evidence: `data/processed/step9/feature_summary.json`, `feature_schemas.json`, and `derived_feature_statistics.csv`. The original Step 7 factory remains available for later comparisons. Step 10 may extend this notebook after this Step 9 section.
